In [20]:
# Keycloak Authorization Code Flow -- Jupyter / Remote Server Edition
# -----------------------------------------------------------------
# Uses an out-of-band code capture: after login Keycloak redirects
# your browser to http://127.0.0.1:{CALLBACK_PORT}/callback?code=...
# That page shows "connection refused" (nothing listens there on the
# server), but the full URL with ?code= is visible in the address bar.
# Paste that URL into Cell 4 to complete the flow.
#
# Requirements: Python 3.7+  (no third-party packages needed)

import base64
import hashlib
import json
import os
import secrets
import urllib.parse
import urllib.request
import urllib.error
from IPython.display import display, HTML

# ------------------------------------------------------------------
# CONFIG -- edit these values before running
# ------------------------------------------------------------------
KEYCLOAK_URL  = "https://lrllkeyclock.azurewebsites.net"
REALM         = "LowResourceLanguageDataTrust"
CLIENT_ID     = "LRLL-API-ID"
SCOPES        = "openid profile email"
CALLBACK_PORT = 8081   # must match the redirect URI registered in Keycloak
# ------------------------------------------------------------------


In [21]:
# -- PKCE helpers --------------------------------------------------

def generate_code_verifier() -> str:
    return base64.urlsafe_b64encode(os.urandom(32)).rstrip(b"=").decode()

def generate_code_challenge(verifier: str) -> str:
    digest = hashlib.sha256(verifier.encode()).digest()
    return base64.urlsafe_b64encode(digest).rstrip(b"=").decode()


In [22]:
# -- Step 1: Generate PKCE values and display the login link -----

redirect_uri   = f"http://127.0.0.1:{CALLBACK_PORT}/callback"
code_verifier  = generate_code_verifier()
code_challenge = generate_code_challenge(code_verifier)
state          = secrets.token_urlsafe(16)

auth_url = (
    f"{KEYCLOAK_URL}/realms/{REALM}/protocol/openid-connect/auth?"
    + urllib.parse.urlencode({
        "client_id":             CLIENT_ID,
        "redirect_uri":          redirect_uri,
        "response_type":         "code",
        "scope":                 SCOPES,
        "state":                 state,
        "code_challenge":        code_challenge,
        "code_challenge_method": "S256",
    })
)

html = (
    "<div style='font-family:sans-serif;padding:12px;border:1px solid #ccc;"
    "border-radius:6px;max-width:700px'>"
    "<b>Step 1 - Log in</b><br><br>"
    f"<a href='{auth_url}' target='_blank' "
    "style='background:#0066cc;color:white;padding:8px 16px;"
    "border-radius:4px;text-decoration:none;font-size:14px'>"
    "&#128274;&nbsp; Click here to log in via Keycloak"
    "</a><br><br>"
    "<small>"
    "After login your browser is redirected to "
    f"<code>http://127.0.0.1:{CALLBACK_PORT}/callback?code=...</code><br>"
    "That page will show a <b>connection error</b> - this is expected.<br>"
    "<b>Copy the full URL</b> from the address bar and paste it into the next cell."
    "</small>"
    "<hr style='margin:10px 0'>"
    "<details>"
    "<summary style='cursor:pointer;font-size:12px'>"
    "Cannot click the button? Expand for the plain URL"
    "</summary>"
    f"<code style='word-break:break-all;font-size:11px'>{auth_url}</code>"
    "</details></div>"
)
display(HTML(html))

print("[*] PKCE code_verifier and state stored in notebook variables.")
print(f"[*] Redirect URI: {redirect_uri}")


[*] PKCE code_verifier and state stored in notebook variables.
[*] Redirect URI: http://127.0.0.1:8081/callback


In [23]:
# -- Step 2: Paste the redirect URL from your browser ------------
#
# After authenticating, your browser is redirected to:
#   http://127.0.0.1:{CALLBACK_PORT}/callback?code=...&state=...
# and shows "connection refused". Copy the FULL URL from the
# browser address bar and paste it at the prompt below.

raw_callback_url = input("Paste the URL Here").strip()

parsed = urllib.parse.urlparse(raw_callback_url)
params = {k: v[0] for k, v in urllib.parse.parse_qs(parsed.query).items()}

if "error" in params:
    raise RuntimeError(
        f"Keycloak returned an error: {params['error']}\n"
        f"Description: {params.get('error_description', 'n/a')}"
    )

if params.get("state") != state:
    raise ValueError(
        "State mismatch -- possible CSRF. "
        "Re-run Cell 3 to generate a fresh login link and try again."
    )

auth_code = params.get("code")
if not auth_code:
    raise ValueError("No 'code' parameter found in the URL. Did you paste the correct URL?")

print(f"[+] Authorization code captured ({auth_code[:12]}...)")


[+] Authorization code captured (81d389e7-221...)


In [24]:
# -- Step 3: Exchange the authorization code for tokens ----------

def exchange_code_for_tokens(code: str, redirect_uri: str, code_verifier: str) -> dict:
    token_url = f"{KEYCLOAK_URL}/realms/{REALM}/protocol/openid-connect/token"
    payload = urllib.parse.urlencode({
        "grant_type":    "authorization_code",
        "client_id":     CLIENT_ID,
        "redirect_uri":  redirect_uri,
        "code":          code,
        "code_verifier": code_verifier,
    }).encode()
    req = urllib.request.Request(
        token_url,
        data=payload,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())


print("[*] Exchanging authorization code for tokens ...")

try:
    tokens = exchange_code_for_tokens(auth_code, redirect_uri, code_verifier)
except urllib.error.HTTPError as e:
    raise RuntimeError(f"Token exchange failed ({e.code}): {e.read().decode()}") from e

sep = "=" * 60
print(f"\n{sep}\n  TOKEN RESPONSE\n{sep}")
print(f"\n  Token type  : {tokens.get('token_type', 'Bearer')}")
print(f"  Expires in  : {tokens.get('expires_in', '?')}s")
print(f"\n  Access token:\n  {tokens.get('access_token', '')[:80]}...")
if tokens.get("id_token"):
    print(f"\n  ID token    :\n  {tokens['id_token'][:80]}...")
if tokens.get("refresh_token"):
    print(f"\n  Refresh token:\n  {tokens['refresh_token'][:80]}...")

token_path = "tokens.json"
with open(token_path, "w") as f:
    json.dump(tokens, f, indent=2)

print(f"\n  Full response saved to: {token_path}")
print(f"{sep}")
print("\n[+] Done. Run  ddplrll-reader run  to query the API.")


[*] Exchanging authorization code for tokens ...

  TOKEN RESPONSE

  Token type  : Bearer
  Expires in  : 300s

  Access token:
  eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJlLXc4MEtuakhZV1M2bTNVcWtjUGV1...

  ID token    :
  eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJlLXc4MEtuakhZV1M2bTNVcWtjUGV1...

  Refresh token:
  eyJhbGciOiJIUzUxMiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJjYjM4ZmRmZS04MDhiLTRkZWQtYjI5...

  Full response saved to: tokens.json

[+] Done. Run  ddplrll-reader run  to query the API.


---
## Query the API

The cells below use `DdplrllDatasetClient` (which automatically reads `tokens.json`) to query **text** and **sound** datasets. Adjust the filter parameters as needed.

In [29]:
# -- Query text datasets -----------------------------------------
#
# Filters (all optional, combinable):
#   keyword  – partial, case-insensitive keyword match
#   theme    – UNBIS theme (e.g. "HEALTH", "EDUCATION")
#   author   – author name
#   year     – publication year as a string (e.g. "2024")
#   limit    – max files returned (1–100, default 30)
#
# Results are saved to:
#   {output_dir}/dataset.jsonld         – Croissant JSON-LD
#   {output_dir}/files/                 – downloaded files (if download=True)
#
# Note: uses await (async method) to work inside Jupyter's event loop.

import json
from ddplrll_reader import DdplrllDatasetClient, Settings

TEXT_OUTPUT = "./output/text"

settings = Settings(
    api_base_url="https://lrllapi.azurewebsites.net",
    output_dir=TEXT_OUTPUT,
    verify_ssl=False
)
client = DdplrllDatasetClient(settings)

# Query + download files + save JSON-LD
jsonld_path = await client.arun(
    keyword="malaria",
    # theme="HEALTH",
    # author="John Banda",
    limit=10,
    output_dir=TEXT_OUTPUT,
    download=True
)

print(f"[+] JSON-LD saved to: {jsonld_path}")

# Preview the sampling info and first file entry
with open(jsonld_path) as fh:
    data = json.load(fh)

sampling = data.get("samplingInfo", {})
print(f"\nMatched: {sampling.get('totalMatched')}  "
      f"Returned: {sampling.get('returned')}  "
      f"Random sample: {sampling.get('randomSample')}")

first_dataset = (data.get("@graph") or data.get("graph") or [None])[0]
if first_dataset:
    files = first_dataset.get("distribution", [])
    print(f"\nDataset : {first_dataset.get('sc:name') or first_dataset.get('scName')}")
    print(f"Files   : {len(files)}")
    if files:
        f0 = files[0]
        print(f"\nFirst file:")
        print(f"  Name    : {f0.get('sc:name') or f0.get('scName')}")
        print(f"  Author  : {f0.get('sc:author') or f0.get('scAuthor')}")
        print(f"  Keywords: {f0.get('sc:keywords') or f0.get('scKeywords')}")
        print(f"  URL     : {f0.get('sc:contentUrl') or f0.get('scContentUrl')}")

[+] JSON-LD saved to: /Users/mozdag/Projects/lowreslanglibapireader/output/text/dataset.jsonld

Matched: 3  Returned: 3  Random sample: False

Dataset : output_extracts_nation_newspaper
Files   : 3

First file:
  Name    : Bungwe lilimbana ndi malungo
  Author  : Patrick Ndawala
  Keywords: ['Malaria', 'NGO', 'Women', 'Collaboration', 'Health campaign']
  URL     : /Users/mozdag/Projects/lowreslanglibapireader/output/text/files/file-output_extracts_nation_newspaper-5fa6de4b.md


In [19]:
# -- Query sound datasets ----------------------------------------
#
# Filters (all optional, combinable):
#   language   – language code or display language (e.g. "ny", "en-GB")
#   programme  – programme / dataset name (partial match)
#   keyword    – keyword match
#   year       – publication year as an integer (e.g. 2024)
#   limit      – max audio files returned (1–100, default 30)
#
# Results are saved to:
#   {output_dir}/sound_dataset.jsonld   – Croissant JSON-LD
#   {output_dir}/files/                 – downloaded audio files (if download=True)
#
# Note: uses await (async method) to work inside Jupyter's event loop.
# file_download_timeout caps the total wall-clock time per file so that
# audio streams that never send EOF are cancelled rather than hanging forever.

import json
from ddplrll_reader import DdplrllDatasetClient, Settings

SOUND_OUTPUT = "./output/sound"

settings = Settings(
    api_base_url="http://localhost:5000",
    output_dir=SOUND_OUTPUT,
    file_download_timeout=120.0,  # seconds; raise if files are large
)
client = DdplrllDatasetClient(settings)

# Query + download audio files + save JSON-LD
sound_path = await client.arun_sound(
    language="ny",
    # programme="Radio Chichewa",
    # keyword="climate",
    year=2024,
    limit=10,
    output_dir=SOUND_OUTPUT,
    download=True,
)

print(f"[+] Sound JSON-LD saved to: {sound_path}")

# Preview the sampling info and first file entry
with open(sound_path) as fh:
    sound_data = json.load(fh)

sampling = sound_data.get("samplingInfo", {})
print(f"\nMatched: {sampling.get('totalMatched')}  "
      f"Returned: {sampling.get('returned')}  "
      f"Random sample: {sampling.get('randomSample')}")

first_dataset = (sound_data.get("@graph") or sound_data.get("graph") or [None])[0]
if first_dataset:
    files = first_dataset.get("distribution", [])
    print(f"\nDataset : {first_dataset.get('sc:name') or first_dataset.get('scName')}")
    print(f"Files   : {len(files)}")
    if files:
        f0 = files[0]
        print(f"\nFirst file:")
        print(f"  Name    : {f0.get('sc:name') or f0.get('scName')}")
        print(f"  Language: {f0.get('sc:inLanguage') or f0.get('scInLanguage')}")
        print(f"  Format  : {f0.get('sc:encodingFormat') or f0.get('scEncodingFormat')}")
        print(f"  URL     : {f0.get('sc:contentUrl') or f0.get('scContentUrl')}")

CancelledError: 